# Personal Finance Cleaning and Analysis

This notebook cleans the messy finance export and turns it into a simpler transaction table.
After that I use the cleaned table to answer a few basic business questions.


## Load the messy dataset

I start by loading the raw sample export. The file is messy because the categories are spread across several columns.


In [1]:
import pandas as pd

df = pd.read_csv("../data/sample/raw_transactions_sample.csv")


In [2]:
df.head(10)


,Unnamed: 0,Food,Unnamed: 2,Other Things,Unnamed: 4,Subscriptions,Unnamed: 6,Housing / Utilities,Transport,Notes,Family,Unnamed: 11,Account Ref,Income,Unnamed: 14
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Demo Export,NaN,NaN,NaN,NaN,NaN
1,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Generated for portfolio only,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-01-03,240.0,Mango Shake Kiosk,300.0,gift loft,NaN,NaN,NaN,240.0,NaN,NaN,NaN,NaN,NaN,NaN
5,2024-01-04,180.0,northside grocer,420.0,Orbit Pharmacy,NaN,NaN,4100,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2024-01-05,80.0,daily roaster,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-01-08,110.0,Spark Soda,1350.0,METRO HOME,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2024-01-09,110.0,Spark Soda,NaN,NaN,NaN,NaN,3800,45.0,NaN,NaN,NaN,NaN,NaN,NaN
9,2024-01-10,250.0,pier pizza,NaN,NaN,NaN,NaN,NaN,45.0,NaN,NaN,NaN,NaN,NaN,NaN


## Inspect the raw data

Before cleaning, I check the columns, missing values, and duplicates.


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           112 non-null    str    
 1   Food                 132 non-null    float64
 2   Unnamed: 2           128 non-null    str    
 3   Other Things         48 non-null     float64
 4   Unnamed: 4           47 non-null     str    
 5   Subscriptions        11 non-null     float64
 6   Unnamed: 6           11 non-null     str    
 7   Housing / Utilities  14 non-null     str    
 8   Transport            55 non-null     float64
 9   Notes                2 non-null      str    
 10  Family               17 non-null     float64
 11  Unnamed: 11          16 non-null     str    
 12  Account Ref          0 non-null      float64
 13  Income               9 non-null      str    
 14  Unnamed: 14          9 non-null      str    
dtypes: float64(6), str(9)
memory usage: 16.1 KB


In [4]:
df.isna().sum().sort_values(ascending=False)


Account Ref            136
Notes                  134
Income                 127
Unnamed: 14            127
Unnamed: 6             125
Subscriptions          125
Housing / Utilities    122
Unnamed: 11            120
Family                 119
Unnamed: 4              89
Other Things            88
Transport               81
Unnamed: 0              24
Unnamed: 2               8
Food                     4
dtype: int64

In [5]:
df.duplicated().sum()


np.int64(3)

In [6]:
df.describe()


,Food,Other Things,Subscriptions,Transport,Family,Account Ref
count,132.000000,48.000000,11.000000,55.000000,17.000000,0.0
mean,336.212121,846.250000,319.727273,159.909091,573.529412,NaN
std,262.812243,651.641544,205.820840,97.778524,307.691834,NaN
min,55.000000,120.000000,120.000000,45.000000,180.000000,NaN
25%,120.000000,420.000000,160.000000,80.000000,340.000000,NaN
50%,250.000000,710.000000,249.000000,160.000000,480.000000,NaN
75%,420.000000,965.000000,455.000000,240.000000,620.000000,NaN
max,980.000000,2600.000000,650.000000,340.000000,1200.000000,NaN


## Rename columns

Some columns came in without useful names, so I rename them first.


In [7]:
df = df.rename(columns={"Unnamed: 0": "Date", "Unnamed: 2":"Food Category", "Unnamed: 4":"Other Things Category", "Unnamed: 6":"Subscription Category", "Unnamed: 11":"Family Category", "Unnamed: 14":"Income Type"})


In [8]:
df.columns


Index(['Date', 'Food', 'Food Category', 'Other Things',
       'Other Things Category', 'Subscriptions', 'Subscription Category',
       'Housing / Utilities', 'Transport', 'Notes', 'Family',
       'Family Category', 'Account Ref', 'Income', 'Income Type'],
      dtype='str')

## Remove columns I do not need

Notes only contains export notes, and Account Ref is empty in this file.


In [9]:
df[["Notes", "Account Ref"]].notna().sum()


Notes          2
Account Ref    0
dtype: int64

In [10]:
df = df.drop(columns=["Notes","Account Ref"])


## Remove empty rows

Some rows are just empty spacing from the export.


In [11]:
df.shape


(136, 13)

In [12]:
df[df.isna().all(axis=1)]


,Date,Food,Food Category,Other Things,Other Things Category,Subscriptions,Subscription Category,Housing / Utilities,Transport,Family,Family Category,Income,Income Type
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df = df.dropna(how="all")


In [14]:
df.shape


(133, 13)

## Clean values and data types

I clean the text values so they are easier to group later. I also convert the money columns that came in as text.


In [15]:
df["Food Category"] = df["Food Category"].str.lower().str.replace(" ", "_", regex=False).str.strip().str.replace("-", "_", regex=False).str.replace(r"\s+", "_", regex=True).str.replace(r"_+", "_", regex=True).str.strip("_")
df["Other Things Category"] = df["Other Things Category"].str.lower().str.replace(" ", "_", regex=False).str.strip().str.replace("-", "_", regex=False).str.replace(r"\s+", "_", regex=True).str.replace(r"_+", "_", regex=True).str.strip("_")
df["Subscription Category"] = df["Subscription Category"].str.lower().str.replace(" ", "_", regex=False).str.strip().str.replace("-", "_", regex=False).str.replace(r"\s+", "_", regex=True).str.replace(r"_+", "_", regex=True).str.strip("_")
df["Housing / Utilities"] = df["Housing / Utilities"].str.replace(",", "", regex=False).str.replace(" ", "", regex=False).astype(float)
df["Family Category"] = df["Family Category"].str.lower().str.replace(" ", "_", regex=False).str.strip().str.replace("-", "_", regex=False).str.replace(r"\s+", "_", regex=True).str.replace(r"_+", "_", regex=True).str.strip("_")
df["Income"] = df["Income"].str.lower().str.replace(",", "", regex=False).astype(float)
df["Income Type"] = df["Income Type"].str.lower().str.replace(" ", "_", regex=False).str.strip().str.replace("-", "_", regex=False).str.replace(r"\s+", "_", regex=True).str.replace(r"_+", "_", regex=True).str.strip("_")


In [16]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


/tmp/ipykernel_352553/1558079315.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [17]:
df.info()


<class 'pandas.DataFrame'>
Index: 133 entries, 1 to 135
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Date                   111 non-null    datetime64[us]
 1   Food                   132 non-null    float64       
 2   Food Category          128 non-null    str           
 3   Other Things           48 non-null     float64       
 4   Other Things Category  47 non-null     str           
 5   Subscriptions          11 non-null     float64       
 6   Subscription Category  11 non-null     str           
 7   Housing / Utilities    14 non-null     float64       
 8   Transport              55 non-null     float64       
 9   Family                 17 non-null     float64       
 10  Family Category        16 non-null     str           
 11  Income                 9 non-null      float64       
 12  Income Type            9 non-null      str           
dtypes: datetime64[us](1),

## Fix missing category values

A few rows have an amount but no category name. I mark those as unknown instead of guessing.


In [18]:
df[df["Family"].notna() & df["Family Category"].isna()]


,Date,Food,Food Category,Other Things,Other Things Category,Subscriptions,Subscription Category,Housing / Utilities,Transport,Family,Family Category,Income,Income Type
129,2024-06-21,580.0,harbor_mart,NaN,NaN,650.0,fit_circle,NaN,240.0,450.0,NaN,NaN,NaN


In [19]:
df.loc[129, "Family Category"] = "unknown"


In [20]:
df[df["Other Things"].notna() & df["Other Things Category"].isna()]


,Date,Food,Food Category,Other Things,Other Things Category,Subscriptions,Subscription Category,Housing / Utilities,Transport,Family,Family Category,Income,Income Type
84,2024-05-01,85.0,spark_soda,450.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23000.0,monthly_payroll


In [21]:
df.loc[84,"Other Things Category"] = "unknown"


In [22]:
df[df["Food"].notna() & df["Food Category"].isna()]


,Date,Food,Food Category,Other Things,Other Things Category,Subscriptions,Subscription Category,Housing / Utilities,Transport,Family,Family Category,Income,Income Type
92,NaT,120.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,NaT,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2024-06-04,120.0,NaN,960.0,orbit_pharmacy,NaN,NaN,NaN,NaN,450.0,family_gift,NaN,NaN
117,2024-06-07,340.0,NaN,NaN,NaN,NaN,NaN,NaN,45.0,NaN,NaN,NaN,NaN


In [23]:
df.loc[[92, 97, 113, 117],"Food Category"] = "unknown"


## Fix the date column

There is still one metadata row, and some transaction rows use a blank date when the date is the same as the row above.
I remove the metadata row, then fill the date down.


In [24]:
df[df["Date"].isna()].head(10)


,Date,Food,Food Category,Other Things,Other Things Category,Subscriptions,Subscription Category,Housing / Utilities,Transport,Family,Family Category,Income,Income Type
1,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,NaT,920.0,harbor_mart,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,NaT,120.0,harbor_mart,610.0,gift_loft,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,NaT,160.0,daily_roaster,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
36,NaT,640.0,northside_grocer,940.0,weekend_trip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,NaT,420.0,fresh_basket,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,NaT,160.0,daily_roaster,390.0,gift_loft,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55,NaT,55.0,spark_soda,390.0,weekend_trip,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
57,NaT,210.0,fresh_basket,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,NaT,210.0,fresh_basket,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
df = df.drop(index=1)


In [26]:
df["Date"] = df["Date"].ffill()


## Inspect the cleaned wide table

The wide table is cleaner now, but it is still not ideal for analysis because each spending type has its own amount column.


In [27]:
df.head(10)


,Date,Food,Food Category,Other Things,Other Things Category,Subscriptions,Subscription Category,Housing / Utilities,Transport,Family,Family Category,Income,Income Type
4,2024-01-03,240.0,mango_shake_kiosk,300.0,gift_loft,NaN,NaN,NaN,240.0,NaN,NaN,NaN,NaN
5,2024-01-04,180.0,northside_grocer,420.0,orbit_pharmacy,NaN,NaN,4100.0,NaN,NaN,NaN,NaN,NaN
6,2024-01-05,80.0,daily_roaster,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2024-01-08,110.0,spark_soda,1350.0,metro_home,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2024-01-09,110.0,spark_soda,NaN,NaN,NaN,NaN,3800.0,45.0,NaN,NaN,NaN,NaN
9,2024-01-10,250.0,pier_pizza,NaN,NaN,NaN,NaN,NaN,45.0,NaN,NaN,NaN,NaN
10,2024-01-11,120.0,weekend_trip,NaN,NaN,NaN,NaN,NaN,240.0,620.0,family_medicine,NaN,NaN
11,2024-01-11,920.0,harbor_mart,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,2024-01-12,120.0,corner_cart,2600.0,pixel_hub,NaN,NaN,3200.0,45.0,NaN,NaN,NaN,NaN
13,2024-01-15,85.0,spark_soda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22000.0,monthly_payroll


In [28]:
df.info()


<class 'pandas.DataFrame'>
Index: 132 entries, 4 to 135
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Date                   132 non-null    datetime64[us]
 1   Food                   132 non-null    float64       
 2   Food Category          132 non-null    str           
 3   Other Things           48 non-null     float64       
 4   Other Things Category  48 non-null     str           
 5   Subscriptions          11 non-null     float64       
 6   Subscription Category  11 non-null     str           
 7   Housing / Utilities    14 non-null     float64       
 8   Transport              55 non-null     float64       
 9   Family                 17 non-null     float64       
 10  Family Category        17 non-null     str           
 11  Income                 9 non-null      float64       
 12  Income Type            9 non-null      str           
dtypes: datetime64[us](1),

In [29]:
df.isna().sum().sort_values(ascending=False)


Income                   123
Income Type              123
Subscriptions            121
Subscription Category    121
Housing / Utilities      118
Family                   115
Family Category          115
Other Things Category     84
Other Things              84
Transport                 77
Food                       0
Date                       0
Food Category              0
dtype: int64

In [30]:
df.shape


(132, 13)

In [31]:
df["Date"].min(), df["Date"].max()


(Timestamp('2024-01-03 00:00:00'), Timestamp('2024-06-28 00:00:00'))

## Reshape the data

I reshape the table into one transaction per row. This makes it easier to group by month, type, and merchant/detail.


In [32]:
long_df_original = pd.concat([df[["Date","Food","Food Category"]].rename(columns={"Food":"Amount", "Food Category":"Category"}).assign(Type="Food"),
df[["Date","Other Things","Other Things Category"]].rename(columns={"Other Things":"Amount","Other Things Category":"Category"}).assign(Type="Other Things"),
df[["Date","Subscriptions","Subscription Category"]].rename(columns={"Subscriptions":"Amount","Subscription Category":"Category"}).assign(Type="Subscriptions"),
df[["Date","Family","Family Category"]].rename(columns={"Family":"Amount","Family Category":"Category"}).assign(Type="Family"),
df[["Date","Income","Income Type"]].rename(columns={"Income":"Amount","Income Type":"Category"}).assign(Type="Income"),
df[["Date","Housing / Utilities"]].rename(columns={"Housing / Utilities": "Amount"}).assign(Type="Housing / Utilities"),
df[["Date","Transport"]].rename(columns={"Transport":"Amount"}).assign(Type="Transport"),
], ignore_index=True
)


In [33]:
long_df_original.head(15)


,Date,Amount,Category,Type
0,2024-01-03,240.0,mango_shake_kiosk,Food
1,2024-01-04,180.0,northside_grocer,Food
2,2024-01-05,80.0,daily_roaster,Food
3,2024-01-08,110.0,spark_soda,Food
4,2024-01-09,110.0,spark_soda,Food
5,2024-01-10,250.0,pier_pizza,Food
6,2024-01-11,120.0,weekend_trip,Food
7,2024-01-11,920.0,harbor_mart,Food
8,2024-01-12,120.0,corner_cart,Food
9,2024-01-15,85.0,spark_soda,Food


In [34]:
long_df_original.shape


(924, 4)

In [35]:
long_df_original.isna().sum()


Date          0
Amount      638
Category    707
Type          0
dtype: int64

In [36]:
long_df = long_df_original.copy()
long_df = long_df.dropna(subset=["Amount"])
long_df = long_df.reset_index(drop=True)


## Inspect the cleaned transaction table

Now I check the final long table before doing the analysis.


In [37]:
long_df.head(15)


,Date,Amount,Category,Type
0,2024-01-03,240.0,mango_shake_kiosk,Food
1,2024-01-04,180.0,northside_grocer,Food
2,2024-01-05,80.0,daily_roaster,Food
3,2024-01-08,110.0,spark_soda,Food
4,2024-01-09,110.0,spark_soda,Food
5,2024-01-10,250.0,pier_pizza,Food
6,2024-01-11,120.0,weekend_trip,Food
7,2024-01-11,920.0,harbor_mart,Food
8,2024-01-12,120.0,corner_cart,Food
9,2024-01-15,85.0,spark_soda,Food


In [38]:
long_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 286 entries, 0 to 285
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      286 non-null    datetime64[us]
 1   Amount    286 non-null    float64       
 2   Category  217 non-null    str           
 3   Type      286 non-null    str           
dtypes: datetime64[us](1), float64(1), str(2)
memory usage: 9.1 KB


In [39]:
long_df.shape


(286, 4)

In [40]:
long_df["Type"].value_counts()


Type
Food                   132
Transport               55
Other Things            48
Family                  17
Housing / Utilities     14
Subscriptions           11
Income                   9
Name: count, dtype: int64

In [41]:
amount_columns = ["Food", "Other Things", "Subscriptions", "Family", "Income", "Housing / Utilities", "Transport"]
wide_transaction_count = df[amount_columns].count().sum()
long_transaction_count = long_df["Amount"].count()

wide_transaction_count, long_transaction_count


(np.int64(286), np.int64(286))

The wide table and long table both show 286 transactions, so the reshape did not lose any transactions.


In [42]:
expenses_df = long_df[long_df["Type"] != "Income"].copy()


## Business Question 1: How much was spent each month?

I grouped expenses by month and ignored income.


In [43]:
monthly_spending = expenses_df[["Date", "Amount"]].copy()
monthly_spending["Month"] = monthly_spending["Date"].dt.to_period("M")
monthly_spending = monthly_spending.groupby("Month", as_index=False)["Amount"].sum()
monthly_spending = monthly_spending.rename(columns={"Amount": "Total Spending"})
monthly_spending


,Month,Total Spending
0,2024-01,31990.0
1,2024-02,24673.0
2,2024-03,14735.0
3,2024-04,26990.0
4,2024-05,30629.0
5,2024-06,33545.0


**Key Finding:** June had the highest spending, with January and May also high.


## Business Question 2: Which categories had the highest total spending?

Here I use Type as the main spending category.


In [44]:
category_spending = expenses_df.groupby("Type")["Amount"].agg(Total="sum", Count="count")
category_spending = category_spending.sort_values("Total", ascending=False)
category_spending


,Total,Count
Type,,
Housing / Utilities,55500.0,14
Food,44380.0,132
Other Things,40620.0,48
Family,9750.0,17
Transport,8795.0,55
Subscriptions,3517.0,11


**Key Finding:** Housing / Utilities was the biggest spending category. Food and Other Things were also large categories.


## Business Question 3: Which merchants appeared most often?

The Category column works like the merchant or detail name for most rows, so I count how often each one appears.


In [45]:
merchant_df = expenses_df[expenses_df["Category"].notna()].copy()
merchant_df = merchant_df.groupby("Category")["Amount"].agg(Count="count", Total="sum")
merchant_df = merchant_df.sort_values("Count", ascending=False)
merchant_df.head(20)


,Count,Total
Category,,
northside_grocer,24,14660.0
fresh_basket,17,5470.0
spark_soda,17,1360.0
pier_pizza,16,6860.0
harbor_mart,12,5880.0
comet_burgers,12,3060.0
luna_cafe,11,2160.0
daily_roaster,10,1200.0
metro_home,8,7290.0


**Key Finding:** northside_grocer appeared most often. fresh_basket, spark_soda, and pier_pizza also appeared many times.


## Business Question 4: What was the average transaction size by category?

I calculate the average amount for each main spending category.


In [46]:
average_df = expenses_df.groupby("Type")["Amount"].agg(Count="count", Total="sum", Average="mean")
average_df["Average"] = average_df["Average"].round(1)
average_df = average_df.sort_values("Average", ascending=False)
average_df


,Count,Total,Average
Type,,,
Housing / Utilities,14,55500.0,3964.3
Other Things,48,40620.0,846.2
Family,17,9750.0,573.5
Food,132,44380.0,336.2
Subscriptions,11,3517.0,319.7
Transport,55,8795.0,159.9


**Key Finding:** Housing / Utilities had the highest average transaction size. Transport had the lowest average transaction size because it had many smaller payments.


## Business Question 5: Which transactions require manual review or cleanup?

I check for missing important fields first. I also check the rows without a merchant/detail name, because those looked suspicious at first.


In [47]:
cleanup_check = expenses_df[
    expenses_df["Date"].isna() |
    expenses_df["Type"].isna() |
    expenses_df["Amount"].isna()
]

cleanup_check


,Date,Amount,Category,Type


In [48]:
missing_detail_check = expenses_df[expenses_df["Category"].isna()][["Date", "Type", "Amount"]]
missing_detail_check["Type"].value_counts()


Type
Transport              55
Housing / Utilities    14
Name: count, dtype: int64

**Key Finding:** Nothing unusual was found in the main cleaned fields. The rows without a merchant/detail name are Housing / Utilities transactions, so they do not look like a cleanup problem.


## Business Question 6: What were the largest individual expenses?

Since the largest rows looked important, I list them separately here.


In [49]:
largest_expenses = expenses_df[["Date", "Category", "Type", "Amount"]].copy()
largest_expenses = largest_expenses.sort_values("Amount", ascending=False)
largest_expenses.head(14)


,Date,Category,Type,Amount
229,2024-06-24,NaN,Housing / Utilities,4500.0
228,2024-06-19,NaN,Housing / Utilities,4500.0
224,2024-04-12,NaN,Housing / Utilities,4500.0
227,2024-05-27,NaN,Housing / Utilities,4200.0
220,2024-01-19,NaN,Housing / Utilities,4200.0
223,2024-04-02,NaN,Housing / Utilities,4200.0
217,2024-01-04,NaN,Housing / Utilities,4100.0
221,2024-02-12,NaN,Housing / Utilities,3800.0
218,2024-01-09,NaN,Housing / Utilities,3800.0
222,2024-02-29,NaN,Housing / Utilities,3800.0


**Key Finding:** The largest individual expenses are Housing / Utilities. These look like normal large household payments, not unusual transactions.


## Final key findings

- Spending was highest in June.
- Housing / Utilities was the largest spending category.
- northside_grocer appeared most often in the merchant/detail column.
- Housing / Utilities had the highest average transaction size.
- Nothing unusual was found in the main cleaned fields.
- The largest individual expenses were Housing / Utilities transactions.
